# Notebook 02 — Global vs Local Tile Capture

**Purpose:** compare a single global-style detector against a tiled Mod30 representation.

Notebook 01 showed:

```text
integers → mod30 residue manifold
constraint gate → 8 persisting residue lanes
```

Notebook 02 asks:

```text
Can one lane capture persisting structure?
Or does persisting structure require local tile union?
```

Paper-facing claim:

```text
A single interpretable feature can be structurally incomplete.
A tiled representation preserves persisting structure by distributing capture across local residue lanes.
```


## 0. Bulletproof setup

Run first. Works locally, from `notebooks/`, in Colab, or as a loose uploaded notebook.


In [ ]:

from pathlib import Path
import sys

def find_repo_root(start=None, marker="src"):
    start = Path.cwd() if start is None else Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / marker).exists():
            return p
    return None

REPO_ROOT = find_repo_root()

if REPO_ROOT is None:
    REPO_ROOT = Path.cwd() / "mod30-manifold-tiling"
    SRC_DIR = REPO_ROOT / "src"
    SRC_DIR.mkdir(parents=True, exist_ok=True)
    (SRC_DIR / "__init__.py").write_text("", encoding="utf-8")
    (SRC_DIR / "mod30.py").write_text("""from math import gcd
MOD30 = 30
MOD30_RESIDUES = [1, 7, 11, 13, 17, 19, 23, 29]
def mod_index(n, mod): return n % mod
def mod_mask(n, residues, mod): return mod_index(n, mod) in residues
def generate_coprime_residues(mod): return [r for r in range(1, mod) if gcd(r, mod) == 1]
def mod30_index(n): return mod_index(n, MOD30)
def mod30_mask(n): return mod_mask(n, MOD30_RESIDUES, MOD30)
def mod30_residues(n_max): return [n for n in range(2, n_max) if mod30_mask(n)]
def single_lane_mask(n, lane=1): return mod30_index(n) == lane
""", encoding="utf-8")
    (SRC_DIR / "tiling_metrics.py").write_text("""from collections import Counter
def lane_counts(values, residue_fn): return dict(Counter(residue_fn(v) for v in values))
def lane_density(values, residue_fn):
    counts = lane_counts(values, residue_fn); total = sum(counts.values())
    return {} if total == 0 else {k: v / total for k, v in sorted(counts.items())}
def gate_summary(values, mask_fn):
    inside = [v for v in values if mask_fn(v)]
    outside = [v for v in values if not mask_fn(v)]
    total = len(values)
    return {"total": total, "inside": len(inside), "outside": len(outside),
            "inside_fraction": len(inside)/total if total else 0.0,
            "outside_fraction": len(outside)/total if total else 0.0}
def lane_coverage(captured_residues, target_residues):
    captured, target = set(captured_residues), set(target_residues)
    hit, missed = captured & target, target - captured
    return {"target_lanes": len(target), "captured_lanes": len(hit), "missed_lanes": len(missed),
            "coverage_fraction": len(hit)/len(target) if target else 0.0,
            "captured_residues": sorted(hit), "missed_residues": sorted(missed)}
""", encoding="utf-8")
    (SRC_DIR / "plots.py").write_text("""import matplotlib.pyplot as plt
def save_current(path, dpi=180):
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    return path
""", encoding="utf-8")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FIGURES_DIR = REPO_ROOT / "figures"
DATA_DIR = REPO_ROOT / "data"
OUTPUTS_DIR = REPO_ROOT / "outputs"
for d in [FIGURES_DIR, DATA_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("src exists:", (REPO_ROOT / "src").exists())


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.mod30 import (
    MOD30,
    MOD30_RESIDUES,
    mod30_index,
    mod30_mask,
    single_lane_mask,
)
from src.tiling_metrics import lane_density, gate_summary, lane_coverage
from src.plots import save_current

print("MOD30_RESIDUES:", MOD30_RESIDUES)


## 1. Build the finite residue dataset

Use the same range as Notebook 01 for easy comparison.


In [ ]:
n_min = 1
n_max = 300
values = np.arange(n_min, n_max + 1)

df = pd.DataFrame({
    "n": values,
    "mod30_residue": [mod30_index(int(n)) for n in values],
})

df["inside_mod30_gate"] = df["n"].apply(lambda n: mod30_mask(int(n)))

# A deliberately narrow detector:
# one local tile pretending to stand for the whole persisting structure.
GLOBAL_LANE = 1
df["single_lane_capture"] = df["n"].apply(lambda n: single_lane_mask(int(n), GLOBAL_LANE))

# Full tiled detector:
df["tiled_capture"] = df["inside_mod30_gate"]

df.head(15)


## 2. Coverage comparison

A one-lane detector is readable but incomplete.  
It captures one persisting lane and misses seven.

A tiled detector captures all eight persisting lanes.


In [ ]:
single_captured_residues = sorted(df.loc[df["single_lane_capture"], "mod30_residue"].unique())
tiled_captured_residues = sorted(df.loc[df["tiled_capture"], "mod30_residue"].unique())

single_coverage = lane_coverage(single_captured_residues, MOD30_RESIDUES)
tiled_coverage = lane_coverage(tiled_captured_residues, MOD30_RESIDUES)

coverage_df = pd.DataFrame([
    {"detector": "single lane: residue 1", **single_coverage},
    {"detector": "local tile union: 8 residues", **tiled_coverage},
])

coverage_df.to_csv(DATA_DIR / "02_global_vs_tiled_coverage.csv", index=False)
coverage_df


In [ ]:
capture_counts = pd.DataFrame({
    "detector": ["single lane", "local tile union"],
    "captured_lanes": [
        single_coverage["captured_lanes"],
        tiled_coverage["captured_lanes"],
    ],
    "missed_lanes": [
        single_coverage["missed_lanes"],
        tiled_coverage["missed_lanes"],
    ],
})

capture_counts


## 3. Plot: lane coverage

This is the simplest paper-facing figure:

```text
single detector → 1/8 coverage
tiled detector  → 8/8 coverage
```


In [ ]:
plt.figure(figsize=(7, 4.5))
plt.bar(capture_counts["detector"], capture_counts["captured_lanes"], label="captured persisting lanes")
plt.bar(capture_counts["detector"], capture_counts["missed_lanes"], bottom=capture_counts["captured_lanes"], alpha=0.35, label="missed persisting lanes")
plt.ylabel("number of persisting mod30 lanes")
plt.title("Global-style single detector vs local tile union")
plt.legend()
save_current(FIGURES_DIR / "06_global_vs_tiled_lane_coverage.png")
plt.show()


## 4. Plot: single lane detector

This intentionally shows what gets lost when a single local tile is treated as global capture.


In [ ]:
outside_single = df[~df["single_lane_capture"]]
inside_single = df[df["single_lane_capture"]]

plt.figure(figsize=(12, 4.8))
plt.scatter(outside_single["n"], outside_single["mod30_residue"], s=10, alpha=0.25, label="not captured")
plt.scatter(inside_single["n"], inside_single["mod30_residue"], s=28, label=f"single lane capture: residue {GLOBAL_LANE}")
plt.yticks(range(30))
plt.xlabel("integer n")
plt.ylabel("n mod 30")
plt.title("Single-lane capture: interpretable but structurally incomplete")
plt.legend()
save_current(FIGURES_DIR / "07_single_lane_capture.png")
plt.show()


## 5. Plot: tiled capture

The full local tile union preserves all eight persisting residue lanes.


In [ ]:
outside_tiled = df[~df["tiled_capture"]]
inside_tiled = df[df["tiled_capture"]]

plt.figure(figsize=(12, 4.8))
plt.scatter(outside_tiled["n"], outside_tiled["mod30_residue"], s=10, alpha=0.25, label="outside gate")
plt.scatter(inside_tiled["n"], inside_tiled["mod30_residue"], s=28, label="local tile union: 8 residues")
plt.yticks(range(30))
plt.xlabel("integer n")
plt.ylabel("n mod 30")
plt.title("Local tile union: persisting Mod30 structure is preserved")
plt.legend()
save_current(FIGURES_DIR / "08_tiled_capture_preserves_structure.png")
plt.show()


## 6. Side-by-side residue counts

This CSV records which residue lanes each detector captures.


In [ ]:
residue_capture_df = pd.DataFrame({
    "residue": range(30),
})
residue_capture_df["is_persisting_lane"] = residue_capture_df["residue"].isin(MOD30_RESIDUES)
residue_capture_df["single_lane_capture"] = residue_capture_df["residue"].eq(GLOBAL_LANE)
residue_capture_df["tiled_capture"] = residue_capture_df["residue"].isin(MOD30_RESIDUES)

residue_capture_df.to_csv(DATA_DIR / "02_residue_capture_map.csv", index=False)
residue_capture_df


In [ ]:
x = residue_capture_df["residue"]

plt.figure(figsize=(12, 4.8))
plt.bar(x - 0.18, residue_capture_df["single_lane_capture"].astype(int), width=0.35, label="single lane")
plt.bar(x + 0.18, residue_capture_df["tiled_capture"].astype(int), width=0.35, label="local tile union")
plt.xticks(range(30))
plt.yticks([0, 1], ["miss", "capture"])
plt.xlabel("residue class mod 30")
plt.ylabel("detector response")
plt.title("Detector map over residue classes: single lane vs tile union")
plt.legend()
save_current(FIGURES_DIR / "09_detector_map_over_residues.png")
plt.show()


## 7. Bridge statement

Notebook 02 isolates the representational issue.

| Representation | Interpretability | Structural coverage |
|---|---:|---:|
| single lane | high | incomplete |
| local tile union | high | complete for Mod30 gate |

Paper-facing phrase:

```text
A single detector can be legible while still missing persisting structure.
The union of local tiles preserves structure without requiring one global feature.
```


## 8. Save compact summary


In [ ]:
summary_md = f"""# Notebook 02 Summary — Global vs Local Tile Capture

Notebook 02 compares one single-lane detector against a full local tile union.

## Result

- Target persisting Mod30 lanes: `{MOD30_RESIDUES}`
- Single-lane detector captures: `{single_coverage["captured_residues"]}`
- Single-lane detector misses: `{single_coverage["missed_residues"]}`
- Single-lane coverage: `{single_coverage["coverage_fraction"]:.3f}`
- Local tile union coverage: `{tiled_coverage["coverage_fraction"]:.3f}`

## Interpretation

A single interpretable feature can be structurally incomplete.
A tiled representation preserves persisting structure by distributing capture across local residue lanes.

## Generated files

- `figures/06_global_vs_tiled_lane_coverage.png`
- `figures/07_single_lane_capture.png`
- `figures/08_tiled_capture_preserves_structure.png`
- `figures/09_detector_map_over_residues.png`
- `data/02_global_vs_tiled_coverage.csv`
- `data/02_residue_capture_map.csv`
"""

summary_path = OUTPUTS_DIR / "02_global_vs_local_tile_capture_summary.md"
summary_path.write_text(summary_md, encoding="utf-8")
print(summary_path)


## 9. Optional: zip-download pattern

Uncomment when running in Colab and you want one downloadable output bundle.


In [ ]:
# Optional zip-download pattern:
#
# import shutil
#
# bundle_name = "notebook_02_global_vs_local_tile_capture_outputs"
# bundle_base = REPO_ROOT / bundle_name
# bundle_zip = REPO_ROOT / f"{bundle_name}.zip"
#
# if bundle_base.exists():
#     shutil.rmtree(bundle_base)
#
# bundle_base.mkdir(parents=True, exist_ok=True)
#
# for folder_name in ["figures", "data", "outputs"]:
#     src_folder = REPO_ROOT / folder_name
#     dst_folder = bundle_base / folder_name
#     if src_folder.exists():
#         shutil.copytree(src_folder, dst_folder)
#
# shutil.make_archive(str(bundle_base), "zip", bundle_base)
# print("Created:", bundle_zip)
#
# # In Google Colab, uncomment:
# # from google.colab import files
# # files.download(str(bundle_zip))


## 10. Recommended Notebook 03

```text
03_sparse_feature_analogy.ipynb
```

Goal:

- build a toy sparse feature matrix over residues
- compare one-hot lane detectors, grouped detectors, and full tile union
- quantify fragmentation / dilution as a representation property
```
